# Borefield visualization export

Exports simulation results from a **Modelon Impact** run of a
`Buildings.Fluid.Geothermal.ZonedBorefields.OneUTube` (or `TwoUTubes`) borefield
model into five files in one export folder, intended as input for the local
ground-temperature visualization tools requested in
[LoneMeertens/modelica-buildings#9](https://github.com/LoneMeertens/modelica-buildings/issues/9)
(horizontal/vertical 2D contour plots and 3D fields, data structured to be
compatible with `geothermsim`/`pygfunction`).

**You only need to set three things below: `WORKSPACE`, `MODEL_PATH`, `RESULT_LABEL`.**
Everything else is discovered from the result file itself.

## Output files (written to `OUTPUT_DIR`, default `./export/`)

All four tables are written in the format selected by `OUTPUT_FORMAT`
(**parquet preferred**, one file per table; read with `pandas.read_parquet()`).
A CSV variant is Optional - Not Advised: at hourly aggregation
`segment_heat_rates.csv` is ~294 MB, over GitHub's 100 MB per-file push limit,
so all export CSVs are git-ignored. Use `Parquet_to_dataframe_or_csv.ipynb` to
regenerate local CSVs from the parquet files.

| File | Content |
|---|---|
| `boreholes.parquet` | One row per borehole: id, zone, (x, y), length, buried depth, radius, tilt/orientation, segment count |
| `borehole_segments.parquet` | One row per borehole × segment: `z_start_m`, `z_end_m` only (x, y, radius are already in `boreholes`) |
| `ground_properties.parquet` | Undisturbed ground temperature profile (`T_undisturbed_K`, `gradient_K_m`, `z_start_gradient`) plus soil thermal properties |
| `segment_heat_rates.parquet` | One row per time-aggregation step × borehole × segment: **energy** exchanged at the borehole wall (J) over that step, plus average power (W) and, where available, zone-level fluid conditions |
| `README.md` | Data dictionary: what every file/column means, units, sign conventions, and the assumptions below, generated from the actual run |

## Key modeling facts baked into this notebook

- `Buildings...ZonedBorefields.OneUTube` simulates **one representative borehole
  per zone**, not one per physical borehole — all boreholes in a zone are
  assumed hydraulically and thermally identical. Per your confirmation, the
  representative zone's segment heat rate is **replicated to every physical
  borehole in that zone**.
- Segment heat rate at the borehole wall is read from
  `<borFie>.groTemRes.QBor_flow[iZon, iSeg]` (W), positive = heat rejected
  from the borehole into the ground. This is the same signal as the internal
  (protected, not present in results) `QBorHol[iZon,iSeg].Q_flow`.
- Segment z-boundaries are **read from the model first** (any per-segment
  length/height array with `nSeg` entries), and only fall back to
  `z_start = dBor + (j-1)*hBor/nSeg` if the model does not expose one — this
  matches the buried-depth offset actually used by the model's own
  thermal-response calculation
  (`Buildings...BaseClasses.HeatTransfer.temperatureResponseMatrix`, which
  positions segment `m` at `dBor + (m-1)*hBor/nSeg`), *not* the unrelated,
  unused, protected `z[i] = hBor/nSeg*(i-0.5)` parameter that only seeds an
  initial-condition default and ignores `dBor`. This keeps the notebook
  correct as-is for today's uniform segmentation and ready for unequal
  segments later without code changes elsewhere.
- Ground temperature profile: `TExt0_start`, `dT_dz`, `z0` (exactly the
  parameters you pointed to).
- `T_fluid_in_zone_K` / `T_fluid_out_zone_K` / `m_flow_zone_kg_s` in
  `segment_heat_rates` are **zone-level**, not per-segment (the model has
  no per-segment fluid state) — named accordingly so they aren't mistaken for
  per-segment values.


In [ ]:
"""Step 1 — Check / install required libraries.

- sdf: reads Modelon/Dymola MAT result files.
- pyarrow: writes segment_heat_rates.parquet (the preferred output format).
"""
import sys, subprocess, importlib
from pathlib import Path

_LIB_TARGET = Path.home() / '.local' / 'impact_libs'

def _ensure_path():
    target = str(_LIB_TARGET)
    if target not in sys.path:
        sys.path.insert(0, target)

def _lib_ok(name):
    _ensure_path()
    try:
        importlib.import_module(name)
        return True
    except ImportError:
        return False

def _ensure_lib(name, no_deps=False):
    if _lib_ok(name):
        print(f'{name} library is available.')
        return
    print(f'Installing {name} into {_LIB_TARGET} ...')
    cmd = [sys.executable, '-m', 'pip', 'install', '--target', str(_LIB_TARGET)]
    if no_deps:
        cmd.append('--no-deps')
    r = subprocess.run(cmd + [name], capture_output=True, text=True)
    if r.returncode == 0:
        _ensure_path()
        print('Installed. If import fails below, restart the kernel and re-run this cell.')
    else:
        print(r.stderr)
        raise RuntimeError(f'Failed to install {name}')

_ensure_lib('sdf', no_deps=True)
_ensure_lib('pyarrow')


In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import sdf


## Configuration

Only these need to change between runs.


In [ ]:
# --- Required -----------------------------------------------------------
WORKSPACE = "test"
MODEL_PATH = "ConferenceCollab_2026.ZonedBorefield_ParallelFlow_LKCC.LKCC_3_MixVolManifold_15Pipes"
RESULT_LABEL = "TestVisualization"

# --- Optional -------------------------------------------------------------
# Exact case to export (e.g. "case_3"). Leave None to auto-pick the most
# recently modified case for the matching experiment.
CASE_NAME = None

# Name of the borefield component inside MODEL_PATH. "borFie" matches every
# LKCC_* model in this workspace; override if a different model names it
# differently.
BORFIE_NAME = "borFie"

# Energy-aggregation bin size for segment_heat_rates, in seconds.
# None -> use the model's own tLoaAgg parameter (<BORFIE_NAME>.tLoaAgg).
AGG_TIME_STEP_S = None

# Output format for the four exported tables (ground_properties, boreholes,
# borehole_segments, segment_heat_rates). Each table is its own file — separate
# files keep one schema per file and let readers load only what they need.
#   "parquet" (preferred) — compact columnar format; segment_heat_rates stays
#             well under GitHub's 100 MB per-file push limit (~18 MB at float64
#             vs ~294 MB as CSV at hourly aggregation). Read with
#             pandas.read_parquet().
#   "csv"     — Optional / NOT ADVISED: segment_heat_rates.csv exceeds GitHub's
#             100 MB per-file limit and will be rejected on push; all export
#             CSVs are git-ignored in this repo for that reason. If CSVs are
#             needed locally, prefer running Parquet_to_dataframe_or_csv.ipynb
#             on the parquet files instead.
#   "both"    — writes both; the CSVs are subject to the same warning.
OUTPUT_FORMAT = "parquet"

# Float precision for the value columns of segment_heat_rates (q_segment_J,
# q_segment_W_avg, T_fluid_in/out_zone_K, m_flow_zone_kg_s). Time columns and
# id columns always keep full precision.
#   "float64" — Pro: bit-exact with the simulation output; safe for numeric
#               regression checks. Con: file roughly 2x the float32 variant
#               (measured ~18.4 MB vs ~8.6 MB for the 15-zone hourly case).
#   "float32" — Pro: roughly halves the file. Con: values rounded to a 24-bit
#               mantissa — relative error up to ~1.2e-7 (mean ~6e-8). For this
#               dataset that is ~2e-5 K on ~285 K temperatures, ~0.5 J on the
#               largest ~8.7e6 J bin energies, ~3e-7 kg/s on flows. Negligible
#               for visualization; NOT bit-exact for regression comparisons.
PARQUET_FLOAT_PRECISION = "float64"

# Should the committed parquet results (export/*.parquet) be git-ignored?
#   None  -> ask interactively when this notebook runs (falls back to "leave
#            .gitignore unchanged" when no interactive input is available,
#            e.g. headless nbconvert execution).
#   True  -> add a managed ignore line to the repo-root .gitignore.
#   False -> remove the managed ignore line (parquet results stay committed).
# NOTE: git-ignoring does NOT untrack files that are already committed — that
# requires `git rm --cached <file>` — and ignored results must then be shared
# out-of-band (OneDrive/SharePoint link or a GitHub Release asset).
GITIGNORE_PARQUET = None

OUTPUT_DIR = Path.cwd() / "export"


## Locate and load the result

Mirrors how Modelon Impact lays out generated results on disk (see
`PostProcess/Resources/mat_to_csv_v1.ipynb` for the same discovery pattern):
`.../workspaces/{workspace}/experiments/{experiment}/cases/{case}/result.mat`,
with a sibling `meta.json` carrying `model_name` and `label`.


In [ ]:
GENERATED = Path('/home/jovyan/impact/generated_resources/workspaces')


def _find_experiments(workspace, model_path, result_label):
    root = GENERATED / workspace / 'experiments'
    if not root.exists():
        raise FileNotFoundError(f'No such workspace results directory: {root}')
    matches = []
    for exp_dir in root.iterdir():
        meta_path = exp_dir / 'meta.json'
        cases_dir = exp_dir / 'cases'
        if not meta_path.exists() or not cases_dir.exists():
            continue
        try:
            meta = json.loads(meta_path.read_text())
        except (json.JSONDecodeError, OSError):
            continue
        if meta.get('model_name', '').strip() != model_path.strip():
            continue
        if meta.get('label', '').strip() != result_label.strip():
            continue
        matches.append((exp_dir, cases_dir))
    return matches


def _list_cases(cases_dir):
    cases = []
    for case_dir in cases_dir.iterdir():
        mat_path = case_dir / 'result.mat'
        if case_dir.is_dir() and mat_path.exists():
            cases.append((case_dir.name, mat_path))
    return cases


def find_result_mat(workspace, model_path, result_label, case_name=None):
    experiments = _find_experiments(workspace, model_path, result_label)
    if not experiments:
        raise FileNotFoundError(
            f'No experiment found for workspace={workspace!r}, '
            f'model_path={model_path!r}, result_label={result_label!r}. '
            'Check WORKSPACE/MODEL_PATH/RESULT_LABEL against the Impact UI.'
        )

    all_cases = []
    for exp_dir, cases_dir in experiments:
        for case_id, mat_path in _list_cases(cases_dir):
            all_cases.append((mat_path.stat().st_mtime, exp_dir, case_id, mat_path))

    if not all_cases:
        raise FileNotFoundError(
            f'Experiment(s) found for {result_label!r} but none have a result.mat yet.'
        )

    if case_name is not None:
        selected = [c for c in all_cases if c[2] == case_name]
        if not selected:
            available = sorted({c[2] for c in all_cases})
            raise FileNotFoundError(
                f'CASE_NAME={case_name!r} not found. Available cases: {available}'
            )
        _, exp_dir, case_id, mat_path = selected[0]
    else:
        all_cases.sort(key=lambda c: c[0], reverse=True)
        _, exp_dir, case_id, mat_path = all_cases[0]
        if len({c[2] for c in all_cases}) > 1:
            print(
                f'Multiple cases found for this result; auto-selected the most '
                f'recently modified: {exp_dir.name}/{case_id}. '
                'Set CASE_NAME to pick a specific one.'
            )

    print(f'Using: {exp_dir.name} / {case_id}')
    print(f'result.mat: {mat_path}')
    return mat_path


MAT_PATH = find_result_mat(WORKSPACE, MODEL_PATH, RESULT_LABEL, CASE_NAME)


In [ ]:
def _flatten(group, prefix=''):
    out = []
    for ds in group.datasets:
        out.append((f'{prefix}.{ds.name}' if prefix else ds.name, ds))
    for g in group.groups:
        out.extend(_flatten(g, f'{prefix}.{g.name}' if prefix else g.name))
    return out


_sdf_data = sdf.load(str(MAT_PATH))
VARS = dict(_flatten(_sdf_data))
print(f'{len(VARS)} variables in result.')


def get_scalar(name, required=True, default=None):
    """Read a parameter's value (scalar or length-1/2 trajectory, Dymola-style)."""
    ds = VARS.get(name)
    if ds is None:
        if required:
            raise KeyError(f'Variable not found in result: {name!r}')
        return default
    return float(np.ravel(ds.data)[0])


def get_trajectory(name):
    ds = VARS.get(name)
    if ds is None:
        raise KeyError(f'Variable not found in result: {name!r}')
    return np.asarray(ds.data, dtype=float)


def find_indexed(base_pattern, index_dims):
    """Collect '<prefix>[i]' or '<prefix>[i,j]' variables into a numpy array.

    base_pattern: e.g. r'borFie\\.groTemRes\\.QBor_flow' (regex-escaped prefix,
    without the bracket suffix).
    index_dims: number of indices, 1 or 2.
    Returns (array of shape given by max indices, per-cell trajectory length may
    vary so scalars vs trajectories are handled by the caller) plus the sorted
    index tuples found.
    """
    if index_dims == 1:
        pat = re.compile(rf'^{base_pattern}\[(\d+)\]$')
    else:
        pat = re.compile(rf'^{base_pattern}\[(\d+),(\d+)\]$')
    found = {}
    for name in VARS:
        m = pat.match(name)
        if m:
            idx = tuple(int(g) for g in m.groups())
            found[idx] = name
    if not found:
        raise KeyError(f'No variables matched pattern {base_pattern}[...]')
    return found


def find_var_matching(*substrings):
    """Best-effort search: names containing ALL given substrings, shortest first."""
    hits = [n for n in VARS if all(s in n for s in substrings)]
    hits.sort(key=len)
    return hits


## Borefield configuration and geometry

Pulled from `<BORFIE_NAME>.borFieDat.conDat.*` (the `LKCC_Configuration` /
`Buildings...Data.Configuration.Template` record) and the borefield component
itself.


In [ ]:
B = BORFIE_NAME
conDat = f'{B}.borFieDat.conDat'

nZon = int(round(get_scalar(f'{B}.nZon', required=False) or get_scalar(f'{conDat}.nZon')))
nSeg = int(round(get_scalar(f'{B}.nSeg')))
hBor = get_scalar(f'{conDat}.hBor')
rBor = get_scalar(f'{conDat}.rBor')
dBor = get_scalar(f'{conDat}.dBor')

# cooBor[nBor, 2] and iZon[nBor] -> natural (numeric) sort of the borehole index.
coo_idx = find_indexed(re.escape(f'{conDat}.cooBor'), 2)
nBor = max(i for i, _ in coo_idx.keys())
x_bor = np.array([get_scalar(coo_idx[(b, 1)]) for b in range(1, nBor + 1)])
y_bor = np.array([get_scalar(coo_idx[(b, 2)]) for b in range(1, nBor + 1)])

iZon_idx = find_indexed(re.escape(f'{conDat}.iZon'), 1)
iZon = np.array([int(round(get_scalar(iZon_idx[(b,)]))) for b in range(1, nBor + 1)])

print(f'nZon={nZon}  nBor={nBor}  nSeg={nSeg}  hBor={hBor}  rBor={rBor}  dBor={dBor}')


## Segment depth boundaries

Look for a per-segment length/height array with `nSeg` entries directly in the
result first (so this notebook automatically picks up unequal segmentation if
the model is extended to expose one later). Today's
`Buildings...ZonedBorefields` models don't publish one (the internal `z[nSeg]`
midpoint parameter is protected and optimized out of results), so this falls
back to `z_start = dBor + (j-1)*hBor/nSeg`, `z_end = dBor + j*hBor/nSeg` —
an equal split of the active length `hBor`, offset by the buried depth
`dBor`. This matches `dBor + (m-1)*hBor/nSeg` in
`Buildings...BaseClasses.HeatTransfer.temperatureResponseMatrix`, which is
the segment position actually used by the model's thermal-response
calculation (the ground-facing physics), as opposed to the protected,
unused `z[i] = hBor/nSeg*(i-0.5)` in `PartialStorage`, which omits `dBor`
because it only seeds an initial-condition default.


In [ ]:
def get_segment_boundaries():
    # 1) Look for an explicit per-segment length/height/z array of size nSeg.
    candidates = []
    for base in (f'{B}.zSeg', f'{B}.hSeg', f'{B}.lSeg', f'{B}.z', f'{conDat}.zSeg', f'{conDat}.hSeg'):
        try:
            idx = find_indexed(re.escape(base), 1)
        except KeyError:
            continue
        if len(idx) == nSeg:
            candidates.append((base, idx))

    if candidates:
        base, idx = candidates[0]
        z_mid = np.array([get_scalar(idx[(j,)]) for j in range(1, nSeg + 1)])
        print(f'Using per-segment array found in the result: {base}[1..{nSeg}]')
        # Reconstruct boundaries from midpoints: boundary_j = midpoint of z_mid[j], z_mid[j+1];
        # anchor the first boundary at dBor and the last at dBor + hBor.
        z_start = np.empty(nSeg)
        z_end = np.empty(nSeg)
        z_start[0] = dBor
        for j in range(nSeg - 1):
            boundary = 0.5 * (z_mid[j] + z_mid[j + 1])
            z_end[j] = boundary
            z_start[j + 1] = boundary
        z_end[-1] = dBor + hBor
        return z_start, z_end

    # 2) Fallback: equal split of the active length hBor, offset by the buried
    #    depth dBor, matching dBor + (m-1)*hBor/nSeg in
    #    BaseClasses.HeatTransfer.temperatureResponseMatrix (the segment
    #    position actually used by the ground thermal-response calculation).
    print(f'No per-segment array found in the result; falling back to an equal '
          f'split of hBor={hBor} into nSeg={nSeg} segments, offset by dBor={dBor}.')
    edges = dBor + np.linspace(0.0, hBor, nSeg + 1)
    return edges[:-1], edges[1:]


Z_START, Z_END = get_segment_boundaries()
print('segment z_start:', np.round(Z_START, 3))
print('segment z_end:  ', np.round(Z_END, 3))


## `ground_properties` table

In [ ]:
soiDat = f'{B}.borFieDat.soiDat'

T_undisturbed_K = get_scalar(f'{B}.TExt0_start')
gradient_K_m = get_scalar(f'{B}.dT_dz')
z_start_gradient = get_scalar(f'{B}.z0')
k_soil_W_mK = get_scalar(f'{soiDat}.kSoi', required=False)
rho_soil_kg_m3 = get_scalar(f'{soiDat}.dSoi', required=False)
c_soil_J_kgK = get_scalar(f'{soiDat}.cSoi', required=False)

ground_df = pd.DataFrame([{
    'T_undisturbed_K': T_undisturbed_K,
    'gradient_K_m': gradient_K_m,
    'z_start_gradient': z_start_gradient,
    'k_soil_W_mK': k_soil_W_mK,
    'rho_soil_kg_m3': rho_soil_kg_m3,
    'c_soil_J_kgK': c_soil_J_kgK,
}])
ground_df


## `boreholes` table

Vertical boreholes only (`tilt_deg=0`), matching the LKCC configuration
(`LKCC_Configuration.mo` does not set a tilt/orientation). Zones are labeled
`Zone_{iZon:02d}`.


In [ ]:
zone_id_of = lambda z: f'Zone_{z:02d}'
borehole_id_of = lambda b: f'BH{b:03d}'

boreholes_df = pd.DataFrame({
    'borehole_id': [borehole_id_of(b) for b in range(1, nBor + 1)],
    'zone_id': [zone_id_of(z) for z in iZon],
    'x_m': x_bor,
    'y_m': y_bor,
    'H_m': hBor,
    'D_m': dBor,
    'r_b_m': rBor,
    'tilt_deg': 0.0,
    'orientation_deg': 0.0,
    'n_segments': nSeg,
})
boreholes_df.head()


## `borehole_segments` table

Only `z_start_m`/`z_end_m` per borehole segment: (x, y), radius, and length
are constant along a (vertical) borehole and already in the `boreholes` table
(length is `H_m / n_segments` there, or per-segment via `z_end_m - z_start_m`
here if segments become unequal).


In [ ]:
seg_rows = []
for b in range(1, nBor + 1):
    for j in range(1, nSeg + 1):
        seg_rows.append({
            'borehole_id': borehole_id_of(b),
            'segment_id': j,
            'z_start_m': Z_START[j - 1],
            'z_end_m': Z_END[j - 1],
        })
borehole_segments_df = pd.DataFrame(seg_rows)
borehole_segments_df.head()


## `segment_heat_rates` table — energy per aggregation step

`<BORFIE_NAME>.groTemRes.QBor_flow[iZon, iSeg]` (W) is the heat rate at the
borehole wall of the **representative borehole of each zone**, positive =
heat rejected from the borehole into the ground.

Rather than reconstructing per-bin energy by trapezoidally integrating this
(possibly coarsely-sampled) heat-rate trajectory, the export reads
`<BORFIE_NAME>.groTemRes.U[k,1]` — the accumulated-heat state the model
itself continuously integrates (`der(U) = QBor_flow`, solved by the
integrator, not reconstructed after the fact). Each aggregation bin's energy
is the **increment of `U` across that bin only**, `U(t_end) - U(t_start)` —
the heat exchanged during that block alone, never the
fully-accumulated-since-start value — plus the average power (W) over the
bin for convenience. `U` is used instead of the model's own per-block
`QAgg_flow`/`U_old` because those are `discrete` and not saved to the result
file; `U` is a continuous trajectory that is saved, and this bin-difference
approach is exact whenever a bin edge lands on a saved output point
(guaranteed at every `tLoaAgg` boundary, since the model generates a time
event there — so this is exact by default, when `AGG_TIME_STEP_S` matches
the model's own `tLoaAgg`). The representative zone value is replicated to
every physical borehole in that zone (per the modeling assumption above).

Fluid conditions are named `T_fluid_in_zone_K`, `T_fluid_out_zone_K`,
`m_flow_zone_kg_s` because they are **zone-level, not per-segment** (the
borefield model does not expose per-segment fluid states) — every segment row
of a given zone/time repeats the same zone value. Best-effort:
`m_flow_zone_kg_s` comes from the generic `<BORFIE_NAME>.m_flow[iZon]`
output; `T_fluid_in_zone_K`/`T_fluid_out_zone_K` are looked up from
`TZoneIn{iZon}.T` / `TZoneOut{iZon}.T` sensors if the top-level model has them
(true for the LKCC_* models), else left as `NaN`.


In [ ]:
time = get_trajectory('time')
t_max = time[-1]

agg_dt = AGG_TIME_STEP_S if AGG_TIME_STEP_S is not None else get_scalar(f'{B}.tLoaAgg')
bin_edges = np.arange(0.0, t_max + agg_dt, agg_dt)
bin_edges[-1] = min(bin_edges[-1], t_max)
bin_edges = np.unique(bin_edges)
n_bins = len(bin_edges) - 1
print(f'Aggregation step: {agg_dt} s -> {n_bins} bins over [0, {t_max}] s')


def bin_energy_and_avg_power(t, u):
    """Energy (J) and mean power (W) of each [t_k, t_{k+1}) bin from the
    accumulated-heat state u (der(u) = QBor_flow, continuously integrated by
    the solver). Energy per bin is the increment of u across that bin only
    (u(t_end) - u(t_start)), never the fully-accumulated-since-start value.
    """
    u_edges = np.interp(bin_edges, t, u)
    energies = np.diff(u_edges)
    avg_power = energies / np.diff(bin_edges)
    return energies, avg_power


qbor_idx = find_indexed(re.escape(f'{B}.groTemRes.QBor_flow'), 2)

# Zone-level fluid conditions (best-effort).
m_flow_idx = find_indexed(re.escape(f'{B}.m_flow'), 1)
t_in_names = {int(re.match(r'TZoneIn(\d+)\.T$', n).group(1)): n
              for n in VARS if re.match(r'TZoneIn(\d+)\.T$', n)}
t_out_names = {int(re.match(r'TZoneOut(\d+)\.T$', n).group(1)): n
               for n in VARS if re.match(r'TZoneOut(\d+)\.T$', n)}
if not t_in_names or not t_out_names:
    print('TZoneIn{i}.T / TZoneOut{i}.T not found; T_fluid_in_zone_K/T_fluid_out_zone_K will be NaN.')

bin_t_repr = 0.5 * (bin_edges[:-1] + bin_edges[1:])

zone_cache = {}
for z in range(1, nZon + 1):
    m_flow_z = (np.interp(bin_t_repr, time, get_trajectory(m_flow_idx[(z,)]))
                if (z,) in m_flow_idx else np.full(n_bins, np.nan))
    t_in_z = (np.interp(bin_t_repr, time, get_trajectory(t_in_names[z]))
              if z in t_in_names else np.full(n_bins, np.nan))
    t_out_z = (np.interp(bin_t_repr, time, get_trajectory(t_out_names[z]))
               if z in t_out_names else np.full(n_bins, np.nan))
    zone_cache[z] = (m_flow_z, t_in_z, t_out_z)

zone_seg_energy = {}   # (iZon, iSeg) -> (energy_J[n_bins], avg_power_W[n_bins])
for (z, s), _ in qbor_idx.items():
    # k flattens (zone, segment) the same way GroundTemperatureResponse.mo
    # does internally: QBor_flow_1d[(i-1)*nSeg+j] <-> QBor_flow[i,j], and
    # U[:,1] is accumulated over that same flattened index.
    k = (z - 1) * nSeg + s
    u = get_trajectory(f'{B}.groTemRes.U[{k},1]')
    zone_seg_energy[(z, s)] = bin_energy_and_avg_power(time, u)

print(f'Aggregated {len(zone_seg_energy)} zone-segment heat-rate signals.')


In [ ]:
rows = []
for b in range(1, nBor + 1):
    z = int(iZon[b - 1])
    m_flow_z, t_in_z, t_out_z = zone_cache[z]
    for s in range(1, nSeg + 1):
        energy_J, avg_power_W = zone_seg_energy[(z, s)]
        for k in range(n_bins):
            rows.append({
                'time_s': bin_edges[k],
                'time_end_s': bin_edges[k + 1],
                'borehole_id': borehole_id_of(b),
                'segment_id': s,
                'zone_id': zone_id_of(z),
                'q_segment_J': energy_J[k],
                'q_segment_W_avg': avg_power_W[k],
                'T_fluid_in_zone_K': t_in_z[k],
                'T_fluid_out_zone_K': t_out_z[k],
                'm_flow_zone_kg_s': m_flow_z[k],
            })

segment_heat_rates_df = pd.DataFrame(rows)
segment_heat_rates_df.head()


## `README.md` — data dictionary for whoever receives this export

Generated from the actual run so the numbers in it always match the CSVs.


In [ ]:
from datetime import datetime, timezone

generated_at = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')

_ext = ('.parquet' if OUTPUT_FORMAT == 'parquet'
        else '.csv' if OUTPUT_FORMAT == 'csv'
        else '.parquet (+ .csv)')

readme = f"""# Borefield visualization export — data dictionary

Generated: {generated_at}
Source model: `{MODEL_PATH}`
Result label: `{RESULT_LABEL}` (workspace `{WORKSPACE}`)
Result file: `{MAT_PATH}`

Exported for local ground-temperature visualization
([LoneMeertens/modelica-buildings#9](https://github.com/LoneMeertens/modelica-buildings/issues/9)),
structured to be compatible with `geothermsim`/`pygfunction`.

## Borefield summary

- {nZon} zone(s), {nBor} borehole(s) total, {nSeg} segment(s) per borehole
- Borehole active length H = {hBor} m, buried depth D = {dBor} m, radius r_b = {rBor} m
- Heat-rate energy aggregation step: {agg_dt:g} s
  ({'model default tLoaAgg' if AGG_TIME_STEP_S is None else 'user override AGG_TIME_STEP_S'})

## Coordinate & sign conventions

- x, y: horizontal Cartesian coordinates, meters, from the model's `cooBor` (borefield-local origin).
- z: depth below grade, positive downward, meters. z=0 is the ground surface.
  A segment's borehole wall sits at [z_start_m, z_end_m], with z_start_m = D + (segment_index-1)*H/n_segments.
- Heat rate / energy: positive = heat rejected from the borehole into the ground
  (the sign of `<borFie>.groTemRes.QBor_flow`, which feeds the ground thermal response).

## Files

All four tables are committed as **Parquet** — read with
`pandas.read_parquet('<table>.parquet')` (requires `pyarrow`). Each table is a
separate file so every file keeps a single schema and readers can load only
what they need. CSV variants are Optional - Not Advised: segment_heat_rates.csv
is ~294 MB at hourly aggregation, over GitHub's 100 MB per-file push limit, and
all export CSVs are git-ignored in this repo. Run
`Parquet_to_dataframe_or_csv.ipynb` (next to the export notebook) to produce
local CSV copies of all four tables. Rows in segment_heat_rates are sorted by
(zone_id, segment_id, time_s, borehole_id) for compression efficiency.

### boreholes{_ext} — one row per borehole

| column | meaning |
|---|---|
| borehole_id | unique borehole id, e.g. BH001 |
| zone_id | independent hydraulic/thermal zone this borehole belongs to |
| x_m, y_m | horizontal position |
| H_m | active borehole length |
| D_m | buried depth (offset from grade to the top of the active length) |
| r_b_m | borehole radius |
| tilt_deg, orientation_deg | 0 for all current models (vertical boreholes) |
| n_segments | number of vertical segments this borehole is discretized into |

### borehole_segments{_ext} — one row per borehole x segment

| column | meaning |
|---|---|
| borehole_id | joins to boreholes |
| segment_id | 1-based index along the borehole, from the top (near D) down |
| z_start_m, z_end_m | depth range of this segment; (x, y, r_b) are constant along the borehole and live in boreholes |

### ground_properties{_ext} — one row, whole-domain soil properties

| column | meaning |
|---|---|
| T_undisturbed_K | undisturbed ground temperature above z_start_gradient |
| gradient_K_m | vertical temperature gradient below z_start_gradient (K/m) |
| z_start_gradient | depth (m) below which the gradient applies; above it, T = T_undisturbed_K |
| k_soil_W_mK | soil thermal conductivity |
| rho_soil_kg_m3 | soil density |
| c_soil_J_kgK | soil specific heat capacity |

### segment_heat_rates{_ext} — one row per aggregation step x borehole x segment

| column | meaning |
|---|---|
| time_s, time_end_s | start/end of the aggregation bin |
| borehole_id, segment_id | join to borehole_segments |
| zone_id | hydraulic/thermal zone |
| q_segment_J | **energy** exchanged at this segment's borehole wall during [time_s, time_end_s), taken as the increment over that bin only of `<borFie>.groTemRes.U[k,1]` (the model's own continuously-integrated accumulated-heat state, der(U) = QBor_flow) — not a trapezoidal reconstruction from the heat-rate trajectory, and not the value accumulated since the start of the simulation |
| q_segment_W_avg | q_segment_J / (time_end_s - time_s), for convenience |
| T_fluid_in_zone_K, T_fluid_out_zone_K | zone-level (not per-segment) fluid inlet/outlet temperature, sampled at bin midpoint |
| m_flow_zone_kg_s | zone-level (not per-segment) mass flow rate, sampled at bin midpoint |

## Modeling assumptions

- `{B}` (`Buildings.Fluid.Geothermal.ZonedBorefields.OneUTube`/`TwoUTubes`) simulates
  **one representative borehole per zone**. All boreholes within a zone are assumed
  hydraulically and thermally identical, so the representative zone's segment heat
  rate is replicated to every physical borehole in that zone in segment_heat_rates.
- Segment z-boundaries are read from a per-segment array in the result if the model
  exposes one; otherwise computed as D + (segment_index-1)*H/n_segments, matching the
  segment position actually used by the model's ground thermal-response calculation
  (`Buildings...BaseClasses.HeatTransfer.temperatureResponseMatrix`).
"""

print(readme)


## Write the export files

All four tables (`ground_properties`, `boreholes`, `borehole_segments`,
`segment_heat_rates`) are written in the format selected by `OUTPUT_FORMAT` in
the configuration cell, one file per table, plus `README.md`. **Parquet is
preferred**; CSV is Optional - Not Advised because `segment_heat_rates.csv`
exceeds GitHub's 100 MB per-file push limit at hourly aggregation (all export
CSVs are git-ignored). Use `Parquet_to_dataframe_or_csv.ipynb` to regenerate
local CSVs from the parquet files.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Clean float noise in the bin edges (e.g. 3599.9999999999995 -> 3600.0) so the
# time columns join cleanly and encode compactly.
heat_out = segment_heat_rates_df.copy()
heat_out['time_s'] = heat_out['time_s'].round(6)
heat_out['time_end_s'] = heat_out['time_end_s'].round(6)

# Sort so zone-replicated rows sit adjacent: the same (zone, segment, time)
# values repeat for every borehole in a zone, and adjacency lets zstd exploit
# the repetition (measured 38.8 -> 23.5 MB from sorting alone at float64).
heat_out = heat_out.sort_values(['zone_id', 'segment_id', 'time_s', 'borehole_id'],
                                kind='mergesort', ignore_index=True)

_FLOAT_VALUE_COLS = ['q_segment_J', 'q_segment_W_avg', 'T_fluid_in_zone_K',
                     'T_fluid_out_zone_K', 'm_flow_zone_kg_s']
if PARQUET_FLOAT_PRECISION == 'float32':
    for c in _FLOAT_VALUE_COLS:
        heat_out[c] = heat_out[c].astype('float32')
elif PARQUET_FLOAT_PRECISION != 'float64':
    raise ValueError(f"PARQUET_FLOAT_PRECISION must be 'float64' or 'float32', "
                     f"got {PARQUET_FLOAT_PRECISION!r}")


def _write_parquet(df, path):
    # zstd level 19 + BYTE_STREAM_SPLIT encoding on float columns: measured
    # 23.5 -> 18.4 MB (float64) / 8.6 MB (float32) on the heat-rates table.
    float_cols = [c for c in df.columns if df[c].dtype.kind == 'f']
    df.to_parquet(path, engine='pyarrow', index=False,
                  compression='zstd', compression_level=19,
                  use_dictionary=False,
                  column_encoding={c: 'BYTE_STREAM_SPLIT' for c in float_cols})


TABLES = [
    ('ground_properties', ground_df),
    ('boreholes', boreholes_df),
    ('borehole_segments', borehole_segments_df),
    ('segment_heat_rates', heat_out),
]

if OUTPUT_FORMAT not in ('parquet', 'csv', 'both'):
    raise ValueError(f"OUTPUT_FORMAT must be 'parquet', 'csv', or 'both', got {OUTPUT_FORMAT!r}")

written = []
if OUTPUT_FORMAT in ('parquet', 'both'):
    for name, df in TABLES:
        _write_parquet(df, OUTPUT_DIR / f'{name}.parquet')
        written.append((f'{name}.parquet', df))
if OUTPUT_FORMAT in ('csv', 'both'):
    print('WARNING (Optional - Not Advised): writing CSV outputs. At hourly '
          'aggregation segment_heat_rates.csv is ~294 MB, which exceeds GitHub\'s '
          '100 MB per-file push limit; all export CSVs are git-ignored in this repo '
          'and cannot be committed. Prefer the parquet outputs, and use '
          'Parquet_to_dataframe_or_csv.ipynb when local CSVs are needed.')
    for name, df in TABLES:
        df.to_csv(OUTPUT_DIR / f'{name}.csv', index=False)
        written.append((f'{name}.csv', df))

(OUTPUT_DIR / 'README.md').write_text(readme)

print(f'Wrote {len(written) + 1} files to {OUTPUT_DIR}:')
for f, df in written:
    size_mb = (OUTPUT_DIR / f).stat().st_size / 1e6
    print(f'  {f:32s} {df.shape[0]:>9,} rows x {df.shape[1]} cols  ({size_mb:,.2f} MB)')
print(f'  {"README.md":32s} data dictionary')


In [ ]:
"""Optional — git-ignore the parquet results?

Controlled by GITIGNORE_PARQUET in the configuration cell:
None = ask here (leave unchanged if no interactive input is available),
True/False = add/remove without asking. Manages a single tagged pair of lines
in the repo-root .gitignore; everything else in that file is left untouched.
"""
_IGNORE_PATTERN = 'PostProcess/Resources/borefield_visualization_export/export/*.parquet'
_TAG = '# managed by Borefield_visualization_export.ipynb (GITIGNORE_PARQUET)'


def _find_repo_root(start):
    for p in [start] + list(start.parents):
        if (p / '.git').exists():
            return p
    return None


_choice = GITIGNORE_PARQUET
if _choice is None:
    try:
        _ans = input('Git-ignore export/*.parquet result files? [y/n, Enter = leave unchanged]: ')
        _ans = _ans.strip().lower()
        _choice = True if _ans in ('y', 'yes') else False if _ans in ('n', 'no') else None
    except Exception:
        print('No interactive input available (headless run); leaving .gitignore unchanged.')
        _choice = None

_root = _find_repo_root(Path.cwd())
if _root is None:
    print('No git repository found above this folder; skipping .gitignore management.')
elif _choice is None:
    print('.gitignore left unchanged.')
else:
    _gi = _root / '.gitignore'
    _lines = _gi.read_text().splitlines() if _gi.exists() else []
    # Drop any existing managed pair (tag line + the pattern line after it).
    _kept, _skip = [], False
    for _l in _lines:
        if _skip:
            _skip = False
            continue
        if _l.strip() == _TAG:
            _skip = True
            continue
        _kept.append(_l)
    if _choice:
        _kept += [_TAG, _IGNORE_PATTERN]
        _gi.write_text('\n'.join(_kept) + '\n')
        print(f'Added to {_gi}:\n  {_IGNORE_PATTERN}')
        print('NOTE: this does not untrack parquet files already committed — use '
              '`git rm --cached <file>` for those. Ignored results must be shared '
              'out-of-band (OneDrive/SharePoint link or a GitHub Release asset).')
    else:
        _gi.write_text('\n'.join(_kept) + ('\n' if _kept else ''))
        print(f'Removed managed parquet ignore line from {_gi} (parquet results stay committed).')
